In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import yaml
import polars as pl
import numpy as np
import torch
import zarr
from tqdm import tqdm

from anngeno import AnnGeno
from scripts import get_burdens, get_correlations, get_burdens_chunky

import multiprocessing

device = "cuda" if torch.cuda.is_available() else "cpu"
num_cores = multiprocessing.cpu_count()
print(device, num_cores)

## Read Anngeno file

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag

## Phenotype GIS plot

In [ ]:
annotation = 'promoterAI' #'am_pathogenicity'
gene_id = 'ENSG00000213398' #'ENSG00000130164' # LDLR

# Get indices for the gene and annotation
gene_idx = gene_id_list.index(gene_id)
annotation_idx = all_annotation_list.index(annotation)

# Get the burden array for the specific gene and annotation
gis_df = pl.DataFrame({
    "sample": sample_id_arr,
    "gis_sum": gene_burdens_sum_df[:, gene_idx, annotation_idx],
    "gis_max": gene_burdens_max_df[:, gene_idx, annotation_idx],
    "gis_top2": gene_burdens_top2_df[:, gene_idx, annotation_idx]
})
gis_df

In [ ]:
trait = "hdl_cholesterol"
# pheno_df = pl.read_parquet("/home/dnanexus/data_dir/phenotypes_corr/ldl_direct_prs_corrected.parquet")
pheno_df = pl.read_parquet(f"/home/dnanexus/data_dir/phenotypes_corr/{trait}_prs_corrected.parquet")
plt_df = gis_df.join(pheno_df, on="sample", how="inner")
plt_df

In [ ]:
from plotnine import *

(
    ggplot(plt_df, aes(x='gis_max', y=f'{trait}_prs_corrected')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=False) +
    labs(
        y=f'{trait} (prs corrected)'
        ) +
    theme_bw()
)

## Debug burden computation

In [ ]:
gt = pl.read_parquet("/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq").filter(pl.col('annotation').str.contains('pLoF'))[['gene_id', 'gene_symbol', 'description']].unique()

gt = gt.with_columns(pl.col("description").str.to_lowercase().alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all("-", "").alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all(" ", "_").alias("phenotype")).drop(['description'])

# PRS corrected phenos available in the ag.phenotypes
ag_phenos = ['apolipoprotein_a', 'apolipoprotein_b', 'cholesterol', 'hdl_cholesterol', 'ldl_direct', 'standing_height', 'triglycerides']

subsest_gt = gt.filter(
    pl.col("gene_id").is_in(ag.annotations.select(pl.col("region")).collect().unique()['region'])
    ).filter(
    pl.col("phenotype").is_in(ag_phenos)
    )

subsest_gt.write_parquet("/home/dnanexus/subset_genes.pq")
subsest_gt

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/subset_genes.pq'

with open(config_path) as f:
    config = yaml.safe_load(f)

print("Loading AnnGeno file")
ag = AnnGeno(filename=config.get("anngeno_file"), filemode="r", low_mem=True)
maf = config.get('maf', 0.001)

variants_to_keep_df = ag.annotations.filter((pl.col('AF_ukb') < maf))
ag.subset_variants(set(variants_to_keep_df.select(pl.col("id")).collect()['id']))

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)
all_annotation_list = list(set(all_annotation_list).intersection(set(ag.annotations.collect_schema().names())))

subsest_gt = pl.read_parquet(associations_df_path)

valid_genes = list(subsest_gt['gene_id'].unique())
n_genes = len(valid_genes)
gene_idx_map = gene_idx_map = {g: i for i, g in enumerate(valid_genes)}
gene_idx_map = gene_idx_map = {g: i for i, g in enumerate(valid_genes)}
n_annos = len(all_annotation_list)
n_samples = ag.sample_count

In [ ]:
%%time

sample_chunk_size = 10_000
gene_chunk_size = n_genes

for start in tqdm(range(0, n_samples, sample_chunk_size), desc=f"Samples: Processing chunks of {sample_chunk_size} samples"):
    end = min(start + sample_chunk_size, n_samples)
    sample_slice = slice(start, end)
    current_sample_ids = ag.samples[sample_slice.start:sample_slice.stop]

    for gene, s_burden, m_burden, t2_burden in get_burdens_chunky.get_burdens_array_streaming(
        ag,
        valid_genes,
        all_annotation_list,
        gene_chunk_size=gene_chunk_size,
        sample_slice=sample_slice,
        device=device,
    ):
        n_samples_in_chunk = s_burden.shape[1]
        annotation_ids = np.array(all_annotation_list)

        sample_col = np.tile(current_sample_ids, n_annos)
        annotation_col = np.repeat(annotation_ids, n_samples_in_chunk)

        df_lazy = pl.LazyFrame({
            "sample_id": sample_col,
            "gene_id": [gene] * (n_samples_in_chunk * n_annos),
            "annotation": annotation_col,
            "sum": s_burden.flatten(),
            "max": m_burden.flatten(),
            "top2": t2_burden.flatten(),
        })

    gc.collect()
    break

## Create and save burdens to zarr

In [ ]:
gt = pl.read_parquet("/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq").filter(pl.col('annotation').str.contains('pLoF'))[['gene_id', 'gene_symbol', 'description']].unique()

gt = gt.with_columns(pl.col("description").str.to_lowercase().alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all("-", "").alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all(" ", "_").alias("phenotype")).drop(['description'])

# PRS corrected phenos available in the ag.phenotypes
ag_phenos = ['apolipoprotein_a', 'apolipoprotein_b', 'cholesterol', 'hdl_cholesterol', 'ldl_direct', 'standing_height', 'triglycerides']

subsest_gt = gt.filter(
    pl.col("gene_id").is_in(ag.annotations.select(pl.col("region")).collect().unique()['region'])
    ).filter(
    pl.col("phenotype").is_in(ag_phenos)
    )

subsest_gt.write_parquet("/home/dnanexus/subset_genes.pq")
subsest_gt

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/subset_genes.pq'
sample_set = set(pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv').select(pl.col('eid').cast(pl.Utf8))['eid'])

output_zarr = "/home/dnanexus/250626_small_anngeno_all_annotations_burdens.zarr"

get_burdens_chunky.compute_and_store_burdens(
    config_path=config_path,
    associations_df_path=associations_df_path,
    output_zarr=output_zarr,
    sample_set=sample_set,
    gene_chunk_size=5,
    sample_chunk_size=5_000,
    device=device,
)

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/subset_genes.pq'
sample_set = set(pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv').select(pl.col('eid').cast(pl.Utf8))['eid'])

output_zarr = "/home/dnanexus/250626_small_anngeno_all_annotations_burdens.zarr"

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    associations_df_path=associations_df_path,
    output_zarr=output_zarr,
    sample_set=sample_set,
    gene_chunk_size=5,
    sample_chunk_size=5_000,
    device=device,
)

### Read zarr

In [ ]:
zarr_burdens_path = '/home/dnanexus/250624_small_anngeno_all_annotations_burdens.zarr'
zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
sample_list = zarr_group["samples"][:]
gene_list = zarr_group["genes"][:]
annotation_list = zarr_group["annotations"][:]

In [ ]:
zarr_group["top2_burdens"][:, :, :].shape

In [ ]:
zarr_group["top2_burdens"][:, :, :]

## Save all correlations